# 04. RAG – Search Tests

In this notebook we verify that our **FAISS index + sentence-transformer embeddings**
can retrieve relevant recipes based on a pantry query.

We will:

1. Load the retrieval pipeline from `rag_pipeline.search`.
2. Run a few test queries (different ingredients / diets / cuisines).
3. Inspect the retrieved recipes (titles, ingredients, steps, nutrition).

In [8]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added: ", project_root)

Project root added:  /Users/biancaleoveanu/CookMate-Recipe-Generator


In [9]:
import json
from typing import List, Optional

import pandas as pd

from rag_pipeline.search import search_recipes

In [10]:
def pretty_print_recipe(r: dict, max_steps: Optional[int] = None):
    """
    Nicely display a retrieved recipe dictionary.
    """
    print("=" * 80)
    print(f"ID: {r.get('recipe_id')} | Title: {r.get('title')}")
    print("-" * 80)

    ing_list = r.get("ingredients_list")
    if ing_list:
        print("Key ingredients:")
        if isinstance(ing_list, list):
            print(", ".join(str(x) for x in ing_list[:10]))
        else:
            print(str(ing_list))
        print()

    steps = r.get("steps_list") or r.get("steps")
    if steps:
        if isinstance(steps, list):
            steps_to_show = steps if max_steps is None else steps[:max_steps]
        else:
            steps_to_show = [s.strip() for s in str(steps).splitlines() if s.strip()]

        print("Steps:")
        for i, step in enumerate(steps_to_show, start=1):
            print(f"{i}. {step}")
        print()

    bits = []
    if r.get("calories") is not None:
        bits.append(f"{r['calories']:.0f} kcal")
    if r.get("protein") is not None:
        bits.append(f"{r['protein']:.1f} g protein")
    if r.get("carbs") is not None:
        bits.append(f"{r['carbs']:.1f} g carbs")
    if r.get("fat") is not None:
        bits.append(f"{r['fat']:.1f} g fat")

    if bits:
        print("Nutrition:", " · ".join(bits))
    print()

## 1. Basic search with only ingredients

We start with a simple test: only ingredients, no diet / cuisine filters.

In [14]:
from rag_pipeline.query_builder import build_query

query = build_query(
    ingredients=["chicken", "rice", "onion"],
    diet=None,
    cuisine=None,
)

results_simple = search_recipes(
    query,
    k=5,
)

len(results_simple)

5

In [15]:
for r in results_simple:
    pretty_print_recipe(r, max_steps=5)

ID: 356574 | Title: Best Rice Ever
--------------------------------------------------------------------------------
Key ingredients:
yellow onion, green bell pepper, eggs, Minute Rice, chicken broth, soya sauce, salt, pepper, olive oil

Steps:
1. bring chicken broth to a boil, remove from heat. add minute rice and let stand for 10-15 mins, until water is absorbed, fluff with fork.
2. meanwhile in a non stick skillet,  sautee onions and green pepper with a pinch of salt and pepper in 1 tbsp of olive oil until lightly browned. add mushrooms and sautee until cooked. push all ingredients to the side of the pan and scramble eggs on the other side until cooked. add rice, and soya sauce to taste. enjoy!

Nutrition: 251 kcal · 10.1 g protein · 43.0 g carbs · 3.7 g fat

ID: 237250 | Title: Onion Rice
--------------------------------------------------------------------------------
Key ingredients:
long-grain white rice, beef broth, vegetable broth, salt, butter, yellow onion, fresh parsley

Step

## 2. Adding diet and cuisine constraints

Now we test the same pantry but with:
- `diet="vegetarian"`
- `cuisine="Italian"`

In [16]:
query = build_query(
    ingredients=["tomato", "pasta", "cheese"],
    diet="vegetarian",
    cuisine="Italian",
)

results_diet_cuisine = search_recipes(
    query,
    k=5,
)

len(results_diet_cuisine)

5

In [17]:
for r in results_diet_cuisine:
    pretty_print_recipe(r, max_steps=5)

ID: 217734 | Title: Creamy Basil and Sun-Dried Tomato Vegan Pasta
--------------------------------------------------------------------------------
Key ingredients:
whole wheat pasta, sun-dried tomatoes, firm silken tofu, garlic cloves, fresh basil, salt, broccoli

Steps:
1. Cook pasta.
2. While pasta is cooking, blend tomatoes, tofu, garlic, basil, and salt in a food processor or blender until smooth.
3. Set aside.
4. When pasta is almost done, add broccoli to pasta water and cook for an additional 3 minutes.
5. Drain and return pasta/broccoli to pot.

Nutrition: 549 kcal · 31.2 g protein · 101.3 g carbs · 6.3 g fat

ID: 390820 | Title: Vegan Veggie and &quot;cheese&quot; Pasta (Gluten Free)
--------------------------------------------------------------------------------
Key ingredients:
rotini pasta, cheddar cheese, zucchini, grape tomatoes, yellow onion, olive oil, water

Steps:
1. In a small pan bring 3 or so cups of water to a boil (it is a good idea to gut veggies to preference wh

## 3. Comparing different pantries

We now compare how the search behaves for a few different pantries.

In [19]:
test_queries = [
    (["shrimp", "garlic"], None, "Asian"),
    (["rice", "egg", "soy sauce"], None, "Asian"),
    (["potato", "cheese"], "vegetarian", None),
]

for ingredients, diet, cuisine in test_queries:
    print("#" * 80)
    print("QUERY:")
    print("  ingredients:", ingredients)
    print("  diet:", diet)
    print("  cuisine:", cuisine)
    print("#" * 80)

    query = build_query(ingredients, diet, cuisine)

    res = search_recipes(query, 3)  # 👈 FIXED
    for r in res:
        pretty_print_recipe(r, max_steps=3)


################################################################################
QUERY:
  ingredients: ['shrimp', 'garlic']
  diet: None
  cuisine: Asian
################################################################################
ID: 143779 | Title: Garlic Lover's Shrimp
--------------------------------------------------------------------------------
Key ingredients:
garlic cloves, extra virgin olive oil, butter, crushed red pepper flakes, large shrimp, coarse salt, cracked black pepper

Steps:
1. Process garlic in food processor to finely chop.
2. Heat a large skillet over medium high heat and add oil and butter. Add garlic and crushed pepper flakes to oil and butter. Season shrimp with salt and toss to coat. Add shrimp to the pan and cook, stirring frequently, until pink and heads curl to tails. Add black pepper, to your taste. Serve immediately.

Nutrition: 245 kcal · 23.9 g protein · 4.6 g carbs · 14.3 g fat

ID: 200707 | Title: Garlic-Lover's Shrimp (5 Ww Points)
------------